In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from xgboost import XGBRegressor
# import warnings

# warnings.filterwarnings("ignore")

# # ۱. بارگذاری داده‌ها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# df_raw = pd.read_excel(file_path)
# df_raw['date'] = pd.to_datetime(df_raw['date'])
# df_raw = df_raw.sort_values(by='date')

# all_sensors = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']

# # ۲. جدا سازی دیتای آموزشی (فرض می‌کنیم داده‌های قدیمی‌تر، وضعیت پایه سیستم هستند)
# # اما کل این داده‌ها را برای آموزش استفاده نمی‌کنیم، ابتدا آن‌ها را پالایش می‌کنیم.
# last_date = df_raw['date'].max()
# split_date = last_date - pd.Timedelta(days=20)
# raw_train_set = df_raw[df_raw['date'] <= split_date].copy()
# test_set = df_raw[df_raw['date'] > split_date].copy()

# # ۳. پالایش دیتای آموزش (ساختن Normal Profile)
# # استفاده از DBSCAN برای شناسایی و حذف نقاطی که با رفتار کلی سیستم همخوانی ندارند
# scaler = StandardScaler()
# scaled_train = scaler.fit_transform(raw_train_set[all_sensors])

# dbscan = DBSCAN(eps=0.8, min_samples=10) # پارامترها بسته به چگالی داده شما قابل تنظیم است

# labels = dbscan.fit_predict(scaled_train)

# # فقط داده‌هایی که در کلاستر اصلی هستند (Label != -1) به عنوان "دیتای سالم" انتخاب می‌شوند
# normal_train_set = raw_train_set[labels != -1].copy()

# print(f"تعداد کل داده‌های آموزشی: {len(raw_train_set)}")
# print(f"تعداد داده‌های سالم تایید شده برای آموزش: {len(normal_train_set)}")
# print(f"میزان داده‌های پرت حذف شده: {len(raw_train_set) - len(normal_train_set)} ردیف")

# # ۴. آموزش مدل‌ها بر اساس "فقط داده‌های سالم"
# models = {}
# for target in all_sensors:
#     features = [s for s in all_sensors if s != target]
#     model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
#     model.fit(normal_train_set[features], normal_train_set[target])
#     models[target] = model

# # ۵. پایش انحراف در ۲۰ روز اخیر
# all_errors = pd.DataFrame(index=test_set.index)
# for target in all_sensors:
#     features = [s for s in all_sensors if s != target]
#     preds = models[target].predict(test_set[features])
#     # محاسبه انحراف (تفاضل مقدار واقعی از مقداری که مدلِ "سالم" پیش‌بینی می‌کند)
#     all_errors[f'err_{target}'] = np.abs(test_set[target] - preds)

# # ۶. محاسبه شاخص انحراف کل سیستم (System Deviation Index)
# test_set['System_Deviation_Index'] = all_errors.mean(axis=1)

# # محاسبه آستانه هشدار بر اساس انحرافات دیتای سالم (Training Error Threshold)
# # اگر انحراف در تست، خیلی بیشتر از انحراف در زمان آموزش باشد، یعنی سیستم غیرعادی است.
# train_preds_all = []
# for target in all_sensors:
#     features = [s for s in all_sensors if s != target]
#     train_preds = models[target].predict(normal_train_set[features])
#     train_preds_all.append(np.abs(normal_train_set[target] - train_preds))

# mean_train_error = np.mean(train_preds_all)
# std_train_error = np.std(train_preds_all)
# threshold = mean_train_error + (3*std_train_error) # آستانه ۳ سیگما

# test_set['Is_Anomaly'] = test_set['System_Deviation_Index'] > threshold

# # ۷. ذخیره خروجی
# output_path = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output3.xlsx'
# test_set.to_excel(output_path, index=False)
# print(f"پایش با مدل پالایش شده انجام و در {output_path} ذخیره شد.")

تعداد کل داده‌های آموزشی: 11633
تعداد داده‌های سالم تایید شده برای آموزش: 10946
میزان داده‌های پرت حذف شده: 687 ردیف
پایش با مدل پالایش شده انجام و در outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output3.xlsx ذخیره شد.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from xgboost import XGBRegressor
import os
import time
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

def run_deviation_monitoring():
    """اجرای تحلیل پایش انحراف با مدل پالایش شده و ذخیره خروجی"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # ۱. بارگذاری داده‌ها
    file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
    output_path = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output3.xlsx'
    
    # ایجاد پوشه خروجی
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return None

    all_sensors = ['AssetID_9357', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 
                   'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']

    # ۲. جدا سازی دیتای آموزشی
    print("🔄 مرحله 1: جداسازی داده‌های آموزشی و تست...")
    
    last_date = df_raw['date'].max()
    split_date = last_date - pd.Timedelta(days=20)
    raw_train_set = df_raw[df_raw['date'] <= split_date].copy()
    test_set = df_raw[df_raw['date'] > split_date].copy()
    
    print(f"   داده‌های آموزشی (قبل از پالایش): {len(raw_train_set):,} رکورد")
    print(f"   داده‌های تست (۲۰ روز اخیر): {len(test_set):,} رکورد")

    # ۳. پالایش دیتای آموزش (ساختن Normal Profile)
    print("🔄 مرحله 2: پالایش داده‌های آموزشی با DBSCAN...")
    
    scaler = StandardScaler()
    scaled_train = scaler.fit_transform(raw_train_set[all_sensors])

    dbscan = DBSCAN(eps=0.8, min_samples=10)
    labels = dbscan.fit_predict(scaled_train)

    # فقط داده‌هایی که در کلاستر اصلی هستند به عنوان "دیتای سالم" انتخاب می‌شوند
    normal_train_set = raw_train_set[labels != -1].copy()

    print(f"   تعداد کل داده‌های آموزشی: {len(raw_train_set):,}")
    print(f"   تعداد داده‌های سالم تایید شده برای آموزش: {len(normal_train_set):,}")
    print(f"   میزان داده‌های پرت حذف شده: {len(raw_train_set) - len(normal_train_set):,} ردیف")

    # ۴. آموزش مدل‌ها بر اساس "فقط داده‌های سالم"
    print("🔄 مرحله 3: آموزش مدل‌های XGBoost...")
    
    models = {}
    for target in all_sensors:
        features = [s for s in all_sensors if s != target]
        model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbosity=0)
        model.fit(normal_train_set[features], normal_train_set[target])
        models[target] = model
        print(f"   ✅ مدل برای {target} آموزش داده شد")

    # ۵. پایش انحراف در ۲۰ روز اخیر
    print("🔄 مرحله 4: پایش انحراف در داده‌های تست...")
    
    all_errors = pd.DataFrame(index=test_set.index)
    for target in all_sensors:
        features = [s for s in all_sensors if s != target]
        preds = models[target].predict(test_set[features])
        # محاسبه انحراف (تفاضل مقدار واقعی از مقداری که مدلِ "سالم" پیش‌بینی می‌کند)
        all_errors[f'err_{target}'] = np.abs(test_set[target] - preds)

    # ۶. محاسبه شاخص انحراف کل سیستم (System Deviation Index)
    test_set['System_Deviation_Index'] = all_errors.mean(axis=1)

    # محاسبه آستانه هشدار بر اساس انحرافات دیتای سالم (Training Error Threshold)
    train_preds_all = []
    for target in all_sensors:
        features = [s for s in all_sensors if s != target]
        train_preds = models[target].predict(normal_train_set[features])
        train_preds_all.append(np.abs(normal_train_set[target] - train_preds))

    mean_train_error = np.mean(train_preds_all)
    std_train_error = np.std(train_preds_all)
    threshold = mean_train_error + (3*std_train_error)  # آستانه ۳ سیگما

    test_set['Is_Anomaly'] = test_set['System_Deviation_Index'] > threshold
    test_set['Anomaly_Label'] = test_set['Is_Anomaly'].map({True: '⚠️ Anomaly', False: '✅ Normal'})
    
    # نمایش آمار نهایی
    anomaly_count = test_set['Is_Anomaly'].sum()
    total_count = len(test_set)
    print(f"\n📊 آمار نهایی پایش:")
    print(f"   تعداد کل رکوردهای تست: {total_count:,}")
    print(f"   تعداد رکوردهای ناهنجار: {anomaly_count:,}")
    print(f"   درصد ناهنجاری: {(anomaly_count/total_count)*100:.2f}%")
    print(f"   آستانه هشدار (Threshold): {threshold:.6f}")

    # ۷. ذخیره خروجی
    print("💾 مرحله 5: ذخیره خروجی...")
    
    try:
        test_set.to_excel(output_path, index=False)
        print(f"✅ پایش با مدل پالایش شده انجام و در {output_path} ذخیره شد.")
        print(f"📊 تعداد رکوردهای نهایی: {len(test_set):,}")
        print(f"📋 تعداد ستون‌ها: {len(test_set.columns)}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return test_set

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش انحراف بیرینگ با مدل پالایش شده")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["22:05", "22:06", "22:07"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_deviation_monitoring()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه پایش انحراف بیرینگ با مدل پالایش شده")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه پایش انحراف بیرینگ با مدل پالایش شده
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش انحراف بیرینگ با مدل پالایش شده
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-01 22:05:05
🔄 شروع تحلیل در 2026-07-01 22:05:05
✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: 11,752
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-04 05:16:35
🔄 مرحله 1: جداسازی داده‌های آموزشی و تست...
   داده‌های آموزشی (قبل از پالایش): 11,633 رکورد
   داده‌های تست (۲۰ روز اخیر): 119 رکورد
🔄 مرحله 2: پالایش داده‌های آموزشی با DBSCAN...
   تعداد کل داده‌های آموزشی: 11,633
   تعداد داده‌های سالم تایید شده برای آموزش: 10,946
   میزان داده‌های پرت حذف شده: 687 ردیف
🔄 مرحله 3: آموزش مدل‌های XGBoost...
   ✅ مدل برای AssetID_9357 آموزش داده شد
   ✅ مدل برای AssetID_9343 آموزش داده شد
   ✅ مدل برای AssetID_9358 آموزش داده شد
   ✅ مدل برای AssetID_9359 آموزش داده شد
   ✅ مدل برای AssetID_9360 آموزش داده شد
   ✅ مدل برای Ass